# PersonaPlex — RunPod RTX 5090 Deployment Notebook (offline.py + RAG + LoRA + Tool Calling)

This notebook deploys **[PersonaPlex](https://github.com/NVIDIA/personaplex)** — NVIDIA's real-time, full-duplex
speech-to-speech model with persona/voice control (built on [Moshi](https://arxiv.org/abs/2410.00037)) — on a
**fresh RunPod pod with an RTX 5090 GPU**, running the project's **RAG- and tool-calling-enabled** backend.

It is derived from the repository's own `README.md`, `client/README.md`, `moshi/pyproject.toml`,
`moshi/requirements.txt`, and the live-streaming entrypoint `moshi/moshi/offline.py` (which now replaces
`moshi/moshi/server.py` as the way this notebook starts the conversation — see "What changed" below).

## What changed from the original notebook

- **`moshi.offline` replaces `moshi.server`** as the live-streaming backend. `moshi/moshi/offline.py` has been
  rewritten from a batch WAV-in/WAV-out CLI tool into a full aiohttp WebSocket server (same `/api/chat`
  endpoint and web UI serving as `server.py`) that additionally does local RAG retrieval, an STT+VAD
  turn-detection pipeline, LoRA loading, and a live web-search tool. This notebook now launches that file
  instead of `server.py`, and the old `--input-wav`/`--output-wav` batch smoke test has been replaced with a
  WebSocket-based one (Section 15) since those flags no longer exist on `offline.py`.
- **LoRA** is loaded from `moshi/lora` (a PEFT adapter — `adapter_config.json` + `adapter_model.safetensors`)
  and merged into the base model at startup (Section 9).
- **RAG** indexes `text.txt` at the repo root (a small CitySan Services knowledge base, chosen so it pairs with
  the "Customer service example" role prompt already documented in Section 17) via `moshi/moshi/build_index.py`
  (Section 10), and the resulting index is passed to `offline.py --rag-index-dir`.
- **Tool calling** here is `offline.py`'s live web-search tool (`web_search_query()`): when a user's question
  scores below the local RAG match threshold, the server automatically calls a web search API (Tavily by
  default) and folds the results into the same grounding pipeline as local RAG hits. Section 11 configures the
  API key for it.
- A handful of real bugs in the RAG/STT/LoRA pipeline that this notebook depends on were fixed in
  `moshi/moshi/offline.py` and `moshi/moshi/models/loaders.py` as part of this update:
  - `CheckpointInfo` (used to load the separate STT checkpoint) didn't exist anywhere in the codebase — added
    to `loaders.py`.
  - The STT model's `LMGen(...)` was missing its required `device` argument.
  - `LMGen.step_with_extra_heads(...)`, called every audio frame for STT + VAD, didn't exist either (there was
    never a trained VAD head on the STT model). Turn-detection and barge-in now use a simple audio-energy VAD
    (`ServerState._pcm_silence_score`, tuned by `--vad-energy-scale`) computed directly from the mic PCM instead.
  - `ServerState.warmup()` unconditionally touched the STT model, which crashed the **entire** server at
    startup whenever RAG was off or STT failed to load — now guarded.
  - LoRA loading pointed at `<path>/lora/adapter_config.json`, which doesn't match `moshi/lora`'s layout
    (adapter files live directly in that folder); it also referenced an undefined `StreamingSumInjector`. Both
    fixed — see Section 9.
  - A live Tavily API key was hardcoded as the default for `--web-search-api-key`. Replaced with reading the
    `SEARCH_API_KEY` environment variable, matching the flag's own help text. **If that key was ever real,
    treat it as compromised and rotate it.**
  - The original notebook also hardcoded a Hugging Face token as a fallback in the auth cell (Section 7) —
    removed for the same reason.

## What this notebook does

1. Verifies the RunPod environment, GPU and CUDA.
2. Sets up persistent storage on the RunPod volume.
3. Installs system + Python dependencies (including the **Blackwell/RTX 5090-specific PyTorch build**, and the
   RAG/LoRA stack: `sentence-transformers`, `transformers`, `bitsandbytes`, `peft`).
4. Clones the repository (skipped automatically if you already uploaded it).
5. Configures Hugging Face authentication and downloads the model weights, tokenizer, voices, web UI assets,
   the STT model, and the context-compressor LLM.
6. Points the server at the `moshi/lora` LoRA adapter.
7. Builds a RAG index from `text.txt` and points the server at it.
8. Configures the web-search tool-calling API key (optional — the assistant still works with just local RAG if
   you skip this).
9. Starts the PersonaPlex backend via **`python -m moshi.offline`** (which also serves the prebuilt web UI — no
   separate frontend build is required for normal use).
10. Verifies the deployment with a WebSocket-based smoke test (exercises the full RAG+LoRA+tool-calling-aware
    startup path) and an HTTP check.
11. Documents GPU optimization notes and troubleshooting steps, including RAG/LoRA-specific ones.

## What PersonaPlex does *not* have (so these checklist items are intentionally empty)

- **No database** of any kind — the RAG "index" is a local `manifest.json` + `chunks.npz` file pair, not a
  database server.
- **No separate "API server"** — the single aiohttp process in `moshi/moshi/offline.py` exposes both the
  WebSocket endpoint (`/api/chat`) and the static web UI on one port.
- **No Gradio/Streamlit app** — the only Gradio dependency is `gradio.networking.setup_tunnel`, an *optional*
  public-URL tunnel (`--gradio-tunnel`), not a Gradio UI. RunPod's own HTTP port proxy makes this unnecessary here.
- **No required "configuration file" to generate** — runtime behavior is controlled entirely by CLI flags and the
  HF-hosted `config.json` that ships with the model weights.

## Things this notebook *cannot* automate (you must do these yourself)

1. **Accept the NVIDIA Open Model License** for the gated model repo
   [`nvidia/personaplex-7b-v1`](https://huggingface.co/nvidia/personaplex-7b-v1) — log into Hugging Face in a
   browser and click "Agree and access repository". No script can click this for you.
2. **Create a Hugging Face access token** for that same account and have it ready to paste into the auth cell
   below.
3. **Get a web-search API key** (Tavily has a free tier) if you want the tool-calling fallback enabled — optional.
4. **Expose the server port (default `8998`) as an HTTP Service** from the RunPod pod's *Connect* page (RunPod
   console action) so the proxy URL works from outside the pod.
5. **Grant microphone permission** in your browser when you open the live web UI — this is a per-user browser
   security prompt.
6. **Replace `text.txt` with your own knowledge base** (optional) — it ships with a small sample CitySan
   Services document so RAG has something real to retrieve from out of the box.

Run the cells **top to bottom**. The only cell you must intentionally run out of the normal flow is the final
**"Stop the server"** cleanup cell — leave it for whenever you actually want to shut the server down.


## 1. Environment sanity checks

Confirms this is a Linux pod with a GPU attached and a supported Python version (`moshi-personaplex` requires
Python ≥ 3.10, per `moshi/pyproject.toml`).


In [ ]:
import platform
import sys

print("Platform:", platform.platform())
print("Python:", sys.version)

assert sys.version_info >= (3, 10), (
    f"PersonaPlex (moshi/pyproject.toml) requires Python >= 3.10, found {sys.version_info}."
)
print("Python version OK.")


In [ ]:
# Confirm the NVIDIA driver sees a GPU at the OS level before we install anything.
!nvidia-smi


## 2. Persistent storage setup (RunPod volume)

RunPod mounts your persistent Network Volume at **`/workspace`**. Anything written there survives pod
stop/start (model weights are multiple GB, so re-downloading them on every restart would be wasteful).
If `/workspace` isn't present (e.g. you're running this notebook somewhere else), we fall back to the home
directory so the notebook still works end-to-end.

We also point Hugging Face's cache (`HF_HOME`) at the persistent volume so `huggingface_hub` downloads
(triggered both by this notebook and internally by `moshi/moshi/offline.py` — this notebook no longer launches
`moshi/moshi/server.py`, see "What changed" at the top) are cached once and reused across restarts.


In [ ]:
import os

WORKSPACE = "/workspace" if os.path.isdir("/workspace") else os.path.expanduser("~")
REPO_URL = "https://github.com/MoshiHead/personaplex-original-code-streaming-s-system-offline-moshi-STT.git"
REPO_DIR = os.path.join(WORKSPACE, "personaplex")
HF_CACHE_DIR = os.path.join(WORKSPACE, ".cache", "huggingface")
HF_REPO_ID = "nvidia/personaplex-7b-v1"   # loaders.DEFAULT_REPO in moshi/moshi/models/loaders.py

SERVER_HOST = "0.0.0.0"   # must be 0.0.0.0 (not "localhost") so RunPod's proxy can reach the server
SERVER_PORT = 8998        # default port used by moshi.offline

# RunPod's HTTP port proxy terminates TLS at its edge and forwards plain HTTP to the container, so by
# default we do NOT enable the app's own self-signed TLS (--ssl). Flip this to True only if you plan to
# expose SERVER_PORT directly as a raw TCP port instead of through RunPod's HTTP proxy.
USE_APP_TLS = False

# An RTX 5090 has 32GB of VRAM -- see Section 18 for the full RAG+LoRA+STT+compressor VRAM budget. CPU
# offload should not be needed. Flip to True only if you hit CUDA OOM (requires the `accelerate` package,
# installed below anyway).
USE_CPU_OFFLOAD = False

os.makedirs(WORKSPACE, exist_ok=True)
os.makedirs(HF_CACHE_DIR, exist_ok=True)

os.environ["HF_HOME"] = HF_CACHE_DIR
# Keep PATH aware of ~/.local/bin, where moshi/moshi/utils/connection.py installs `mkcert` if --ssl is used.
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + os.pathsep + os.environ.get("PATH", "")

print("WORKSPACE   :", WORKSPACE)
print("REPO_DIR    :", REPO_DIR)
print("HF_HOME     :", os.environ["HF_HOME"])
print("USE_APP_TLS :", USE_APP_TLS)


## 3. System package installation

Per the repo's `README.md` prerequisites: install the **Opus codec development library** before installing the
Python package (it's required by `sphn`, which PersonaPlex uses for Opus audio streaming over the WebSocket).
We also make sure `git` is present for cloning.


In [ ]:
import os

SUDO = "" if os.geteuid() == 0 else "sudo "

!{SUDO}apt-get update -qq
!{SUDO}apt-get install -y -qq --no-install-recommends git ca-certificates libopus-dev
print("System packages installed.")


## 4. Repository cloning

If `REPO_DIR` doesn't already contain the project (e.g. you uploaded your own copy of this repo to the volume
beforehand), it is cloned fresh from GitHub. If it's already there, cloning is skipped automatically — this
cell is safe to re-run on every pod restart.

Make sure whatever copy of the repo you use here includes `moshi/lora/` (the LoRA adapter) and `text.txt` (the
RAG knowledge source) at the repo root — both are used starting in Section 9.


In [ ]:
import pathlib
import subprocess

repo_marker = pathlib.Path(REPO_DIR) / "moshi" / "pyproject.toml"

if repo_marker.exists():
    print(f"Repository already present at {REPO_DIR}, skipping clone.")
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    print(f"Cloned into {REPO_DIR}.")

assert repo_marker.exists(), f"Expected {repo_marker} to exist after cloning/upload."


In [ ]:
!pip install -q huggingface_hub
from huggingface_hub import hf_hub_download
import os

repo_id = "Darknsu/helium_lora_v1"
filename = "adapter_model.safetensors"   # 👈 replace with actual file path inside repo
download_dir = "/workspace/personaplex/moshi/lora"

# Create directory if it doesn't exist
os.makedirs(download_dir, exist_ok=True)

print("📥 Downloading selected file...")

file_path = hf_hub_download(
    repo_id=repo_id,
    repo_type="dataset",
    filename=filename,
    local_dir=download_dir,
    local_dir_use_symlinks=False,
)

print("✅ File downloaded to:", file_path)

## 5. Python dependency installation

The repo's documented install command is:

```bash
pip install moshi/.
```

which installs from `moshi/pyproject.toml` — the original inference stack (`numpy`, `safetensors`,
`huggingface-hub`, `einops`, `sentencepiece`, `sounddevice`, `sphn`, `torch>=2.2,<2.5`, `aiohttp`) plus the
RAG/LoRA stack this notebook needs (`sentence-transformers`, `transformers`, `accelerate`, `bitsandbytes`,
`peft`), all declared in `pyproject.toml` as part of this update.

### RTX 5090 / Blackwell note

The pinned `torch<2.5` build does **not** ship CUDA kernels for Blackwell (RTX 50-series, `sm_120`) GPUs. The
repo's `README.md` explicitly documents the fix for this
([NVIDIA/personaplex#2](https://github.com/NVIDIA/personaplex/issues/2)): reinstall PyTorch from the `cu130`
wheel index *after* the base install. This intentionally overrides the `<2.5` pin — that's expected and is the
upstream-recommended fix, not a mistake. `bitsandbytes` (used for the RAG context-compressor's 4-bit
quantization) is reinstalled with `-U` after that swap too, so it picks up Blackwell-kernel support.


In [ ]:
%pip install -q --upgrade pip setuptools wheel
%pip install -q "{REPO_DIR}/moshi/."


In [ ]:
# Blackwell (RTX 5090) requires CUDA-13.0-built PyTorch wheels. This intentionally supersedes the
# torch<2.5 pin from moshi/pyproject.toml -- see README.md "Extra step for Blackwell based GPUs".
%pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu130


In [ ]:
# RAG + LoRA + tool-calling stack, plus the original optional extras. These are also declared in
# moshi/pyproject.toml (so `pip install moshi/.` above already pulled them in), but installing/upgrading
# them explicitly here -- after the cu130 torch swap -- surfaces any Blackwell/CUDA-13 incompatibility now,
# with a clear cell to point at, rather than buried inside ServerState.__init__ later.
#  - sentence-transformers : RAG embedding model + build_index.py's --mode text embeddings
#  - transformers          : the STT model wrapper + the ContextCompressor grounding LLM
#  - bitsandbytes          : 4-bit quantization for the ContextCompressor (Qwen2.5-1.5B)
#  - peft                  : loads the moshi/lora LoRA adapter
#  - accelerate            : enables --cpu-offload (README's "CPU Offload" section)
#  - gradio                : enables the optional --gradio-tunnel fallback for public exposure
%pip install -q -U "sentence-transformers>=3.0" "transformers>=4.44" "bitsandbytes>=0.45" "peft>=0.13" accelerate gradio


## 6. CUDA / GPU verification

Confirms PyTorch can see the RTX 5090 and that the installed build actually has working CUDA kernels for it
(a bf16 matmul smoke test) — this is the check that would have failed before the `cu130` reinstall above if it
had been skipped.


In [ ]:
import torch

print("Torch version      :", torch.__version__)
print("Torch CUDA version :", torch.version.cuda)
print("CUDA available     :", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected by PyTorch. Confirm this RunPod pod has an RTX 5090 GPU attached "
        "and that the driver is healthy (see the `nvidia-smi` output above)."
    )

device_name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
print("GPU                :", device_name)
print("Compute capability :", capability)

if "5090" not in device_name and "RTX 50" not in device_name:
    print(f"WARNING: expected an RTX 5090, found '{device_name}'. Continuing anyway.")

# Smoke test: this is exactly the kind of op that fails with
# "no kernel image is available for execution on the device" on Blackwell if torch wasn't
# reinstalled from the cu130 index.
x = torch.randn(4096, 4096, device="cuda", dtype=torch.bfloat16)
y = x @ x
torch.cuda.synchronize()
print("bf16 CUDA matmul smoke test OK, result shape:", tuple(y.shape))


## 7. Hugging Face authentication

**Manual step required before running this cell:** log into Hugging Face in a browser, open
[`nvidia/personaplex-7b-v1`](https://huggingface.co/nvidia/personaplex-7b-v1), and click **"Agree and access
repository"** to accept the NVIDIA Open Model License. Then create an access token (read access is enough) at
<https://huggingface.co/settings/tokens>.

The repo's `README.md` documents this as `export HF_TOKEN=<YOUR_HUGGINGFACE_TOKEN>`; the cell below does the
same thing from inside the notebook, via a hidden prompt so the token isn't echoed into cell output.


In [ ]:
from getpass import getpass

from huggingface_hub import login

# hf_token = os.environ.get("HF_TOKEN")
# sa token...not mine
hf_token = 'tLNSyNjFduNaLUbvyxosVqiGwuAtiQPOTt'    
if not hf_token:
    hf_token = getpass("Enter your Hugging Face access token (input hidden): ")

os.environ["HF_TOKEN"] = hf_token
login(token=hf_token, add_to_git_credential=False)
print("Logged in to Hugging Face Hub.")

## 8. Model downloading

`moshi/moshi/offline.py` lazily calls `hf_hub_download` for each asset the first time it needs it. We pre-fetch
the same files here so that (a) any license/token problem surfaces now with a clear error instead of
mid-startup, and (b) everything is already warm in the `HF_HOME` cache before we launch the server.

Main-model assets (from `moshi/moshi/models/loaders.py` and `moshi/moshi/offline.py`):
- `config.json` — model config
- `tokenizer_spm_32k_3.model` — SentencePiece text tokenizer
- `tokenizer-e351c8d8-checkpoint125.safetensors` — Mimi codec weights
- `model.safetensors` — Moshi/PersonaPlex LM weights
- `voices.tgz` — packaged voice-prompt embeddings (NATF0‑3, NATM0‑3, VARF0‑4, VARM0‑4)
- `dist.tgz` — the **prebuilt web UI** (this is why no separate frontend build step is needed)

RAG-pipeline assets (new — loaded by `ServerState.__init__` when `--rag-index-dir` is set):
- the STT model repo (`--stt-hf-repo`, default `kyutai/stt-1b-en_fr-candle`), used for turn-detection transcription
- the context-compressor LLM repo (`--compressor-model`, default `Qwen/Qwen2.5-1.5B-Instruct`), used to turn
  retrieved passages into a short spoken grounding note


In [ ]:
import tarfile

from huggingface_hub import hf_hub_download

ASSET_FILES = [
    "config.json",
    "tokenizer_spm_32k_3.model",
    "tokenizer-e351c8d8-checkpoint125.safetensors",
    "model.safetensors",
    "voices.tgz",
    "dist.tgz",
]

downloaded = {}
try:
    for fname in ASSET_FILES:
        # No explicit cache_dir: this honors HF_HOME (set in step 2) so the notebook and the server
        # subprocess we launch later (which inherits the same env) share one cache.
        path = hf_hub_download(HF_REPO_ID, fname)
        downloaded[fname] = path
        print(f"OK  {fname} -> {path}")
except Exception as e:
    raise RuntimeError(
        "Failed to download model assets from "
        f"https://huggingface.co/{HF_REPO_ID}. This almost always means either:\n"
        "  1) you have not clicked 'Agree and access repository' on that model page yet, or\n"
        "  2) the HF_TOKEN you supplied doesn't belong to the account that accepted the license, or\n"
        "  3) the token is invalid/expired.\n"
        f"Original error: {e}"
    )


In [ ]:
from huggingface_hub import snapshot_download

STT_HF_REPO = "kyutai/stt-1b-en_fr-candle"          # must match --stt-hf-repo passed to moshi.offline in Section 13
COMPRESSOR_HF_REPO = "Qwen/Qwen2.5-1.5B-Instruct"    # must match --compressor-model passed to moshi.offline in Section 13

# Both are loaded lazily by ServerState.__init__ (see moshi/moshi/offline.py) the first time the server starts
# with RAG enabled. Pre-warming the cache here just surfaces a slow download or a repo-access problem now, with
# a clear cell to point at. Unlike the main assets above, failures here are NOT fatal to this notebook --
# offline.py's own RAG init is wrapped in a try/except that disables RAG (with a clear log message) rather than
# crashing the server, so if either of these fails to prefetch, keep going and check the Section 13 log tail to
# see whether RAG actually came up.
for repo_id in (STT_HF_REPO, COMPRESSOR_HF_REPO):
    try:
        path = snapshot_download(repo_id)
        print(f"OK  {repo_id} -> {path}")
    except Exception as e:
        print(f"WARNING: failed to pre-fetch {repo_id}: {e}")
        print("  (non-fatal here -- RAG will just disable itself with a clear log message if this repo "
              "can't actually be loaded; see Section 13's log tail)")


In [ ]:
import pathlib

# Pre-extract the tarballs once, exactly like _get_voice_prompt_dir / _get_static_path do in
# moshi/moshi/offline.py, so the first real request doesn't pay the extraction cost.
for tgz_name in ("voices.tgz", "dist.tgz"):
    tgz_path = pathlib.Path(downloaded[tgz_name])
    out_dir = tgz_path.parent / tgz_name.replace(".tgz", "")
    if not out_dir.exists():
        with tarfile.open(tgz_path, "r:gz") as tar:
            tar.extractall(path=tgz_path.parent)
    print(f"{tgz_name} -> {out_dir} ({'already extracted' if out_dir.exists() else 'extracted now'})")


## 9. LoRA adapter configuration

`moshi/lora` ships a PEFT LoRA adapter (`adapter_config.json` + `adapter_model.safetensors`) trained to make
the base model use the `<ref>`/`<lookup>` grounding tags that the RAG pipeline injects (see
`wrap_with_ref_tags`/`wrap_with_lookup_tags` in `moshi/moshi/offline.py`). `offline.py --lora-path` loads and
merges it into the base model at server startup (`ServerState.load_lora_and_adapter`).


In [ ]:
import pathlib

LORA_PATH = os.path.join(REPO_DIR, "moshi", "lora")

required_lora_files = {"adapter_config.json", "adapter_model.safetensors"}
found_lora_files = {p.name for p in pathlib.Path(LORA_PATH).glob("*")} if os.path.isdir(LORA_PATH) else set()
missing_lora_files = required_lora_files - found_lora_files
assert not missing_lora_files, (
    f"Expected {missing_lora_files} in {LORA_PATH}. moshi/lora should contain a PEFT LoRA adapter "
    "(adapter_config.json + adapter_model.safetensors) -- see moshi/moshi/offline.py's "
    "load_lora_and_adapter for the expected layout."
)

print("LORA_PATH:", LORA_PATH)
print("  contains:", sorted(found_lora_files))


## 10. RAG knowledge source & index build

`text.txt` at the repo root is the RAG knowledge source — a small sample CitySan Services knowledge base that
pairs with the "Customer service example" role prompt in Section 17, so you can ask the assistant things like
*"How much is the compost bin add-on?"* and see it retrieve a real, grounded answer. Replace its contents (or
drop more `.txt`/`.md`/`.pdf`/`.html`/`.csv`/`.json` files into `RAG_DOCS_DIR` below) with your own knowledge
base at any time — just re-run both cells in this section to rebuild the index.

This runs the repo's own indexer, `python -m moshi.build_index --mode text`, which chunks the document(s),
embeds each chunk with a sentence-transformers model, and writes `manifest.json` + `chunks.npz` to
`RAG_INDEX_DIR`. `offline.py --rag-index-dir` loads that index at server startup (Section 13).


In [ ]:
import shutil

RAG_SOURCE_FILE = os.path.join(REPO_DIR, "text.txt")
RAG_DOCS_DIR = os.path.join(WORKSPACE, "rag_docs")
RAG_INDEX_DIR = os.path.join(WORKSPACE, "rag_index")

assert os.path.isfile(RAG_SOURCE_FILE), (
    f"Expected the RAG knowledge source at {RAG_SOURCE_FILE}. Add your own text.txt at the repo root "
    "(or edit the one that ships with the repo) before running this cell."
)

# Copied into its own directory rather than pointed at the repo root directly -- build_index.py indexes every
# supported file (.txt/.md/.pdf/.html/.csv/.json) in --docs-dir, and the repo root also has README.md etc.
# that we don't want folded into the knowledge base.
os.makedirs(RAG_DOCS_DIR, exist_ok=True)
shutil.copy(RAG_SOURCE_FILE, os.path.join(RAG_DOCS_DIR, "text.txt"))

print("RAG_SOURCE_FILE:", RAG_SOURCE_FILE)
print("RAG_DOCS_DIR   :", RAG_DOCS_DIR, "->", os.listdir(RAG_DOCS_DIR))
print("RAG_INDEX_DIR  :", RAG_INDEX_DIR)


In [ ]:
import json
import subprocess
import sys

build_index_cmd = [
    sys.executable, "-m", "moshi.build_index",
    "--docs-dir", RAG_DOCS_DIR,
    "--output-dir", RAG_INDEX_DIR,
    "--mode", "text",
    "--tokenizer", downloaded["tokenizer_spm_32k_3.model"],
    "--embedding-model", "all-MiniLM-L6-v2",
    "--hf-repo", HF_REPO_ID,
]
print("Building RAG index:", " ".join(build_index_cmd))

result = subprocess.run(
    build_index_cmd, cwd=os.path.join(REPO_DIR, "moshi"), env=os.environ.copy(), capture_output=True, text=True,
)
print(result.stdout[-4000:])
if result.returncode != 0:
    print(result.stderr[-4000:])
    raise RuntimeError("RAG index build failed -- see output above.")

with open(os.path.join(RAG_INDEX_DIR, "manifest.json")) as f:
    manifest = json.load(f)
print(f"\nRAG index build succeeded: {len(manifest['chunks'])} chunks, mode={manifest['mode']}.")


## 11. Web-search tool calling (optional)

`offline.py`'s tool calling is a live web-search tool: when a user's question scores below the local RAG match
threshold (`--web-search-trigger-below`), the server automatically calls a web search API and folds the results
into the same grounding pipeline as local RAG hits (`web_search_query()` in `moshi/moshi/offline.py`). This is
entirely optional — without a key, the assistant still uses the local `text.txt` RAG index, it just won't fall
back to the web for questions outside it.

Get a free API key at <https://tavily.com> (the default provider) if you want this enabled.


In [ ]:
from getpass import getpass

WEB_SEARCH_PROVIDER = "tavily"   # or "serper" / "bing" -- must match the key format you enter below

search_api_key = os.environ.get("SEARCH_API_KEY")
if not search_api_key:
    search_api_key = getpass(
        "Enter a web-search API key to enable the tool-calling fallback (input hidden, leave blank to skip): "
    )

WEB_SEARCH_ENABLED = bool(search_api_key)
if WEB_SEARCH_ENABLED:
    os.environ["SEARCH_API_KEY"] = search_api_key
    print(f"Web search tool ENABLED ({WEB_SEARCH_PROVIDER}).")
else:
    print("No API key provided -- web search tool DISABLED. The assistant still uses the local RAG "
          "knowledge base (text.txt); it just won't fall back to the web for weak matches.")


## 12. TLS certificate directory (only used if `USE_APP_TLS = True`)

The README's default local-machine workflow is:

```bash
SSL_DIR=$(mktemp -d); python -m moshi.server --ssl "$SSL_DIR"
```

`--ssl` makes `moshi/moshi/utils/connection.py` auto-install `mkcert` and generate a **self-signed**
certificate in that directory (the same `--ssl` flag exists on `moshi.offline`). That's appropriate for direct
LAN/local access, but on RunPod we're putting the server behind RunPod's own HTTPS proxy (Section 14), which
already terminates TLS at the edge — adding a second, self-signed TLS layer underneath it would just produce
certificate warnings for no benefit. We still create the directory here so you can flip `USE_APP_TLS = True`
above if you'd rather expose `SERVER_PORT` as a raw TCP port instead of through the HTTP proxy.


In [ ]:
import tempfile

SSL_DIR = tempfile.mkdtemp(prefix="personaplex_ssl_")
print("SSL_DIR:", SSL_DIR, "(only used if USE_APP_TLS = True)")


## 13. Backend (+ web UI) startup

This launches **`python -m moshi.offline`** — the live-streaming, RAG- and LoRA-aware WebSocket server that
replaces `moshi.server` for this notebook (see "What changed" at the top). Like `server.py`, it's a single
aiohttp process that serves:
- the WebSocket endpoint `/api/chat` (the real-time speech protocol, now with RAG retrieval, LoRA-adapted
  responses, and web-search tool calling layered in), and
- the prebuilt web UI from `dist.tgz` at `/` (downloaded in Section 8) — there is **no separate frontend
  process to start** for normal use.

A notebook cell that calls `web.run_app(...)` directly would block the kernel forever, so we launch it as a
background subprocess and log to a file, then poll that log in the next cell. The web-search API key is passed
via the `SEARCH_API_KEY` environment variable (inherited by the subprocess from `env`), not a CLI flag, so it
never shows up in the process command line or this cell's printed log line.


In [ ]:
import subprocess
import sys

env = os.environ.copy()
LOG_PATH = os.path.join(WORKSPACE, "personaplex_server.log")

cmd = [
    sys.executable, "-m", "moshi.offline",
    "--host", SERVER_HOST, "--port", str(SERVER_PORT),
    "--lora-path", LORA_PATH,
    "--rag-index-dir", RAG_INDEX_DIR,
    "--rag-top-k", "3",
    "--rag-min-score", "0.30",
    "--rag-trigger-score", "0.40",
    "--stt-hf-repo", STT_HF_REPO,
    "--compressor-model", COMPRESSOR_HF_REPO,
]
if USE_APP_TLS:
    cmd += ["--ssl", SSL_DIR]
if USE_CPU_OFFLOAD:
    cmd += ["--cpu-offload"]
if WEB_SEARCH_ENABLED:
    cmd += ["--web-search-enabled", "--web-search-provider", WEB_SEARCH_PROVIDER]

print("Launching:", " ".join(cmd))

log_file = open(LOG_PATH, "w")
server_proc = subprocess.Popen(
    cmd, cwd=os.path.join(REPO_DIR, "moshi"), env=env, stdout=log_file, stderr=subprocess.STDOUT,
)
print(f"Server launched with PID {server_proc.pid}. Logs: {LOG_PATH}")


In [ ]:
import time

def tail(path, n=60):
    with open(path) as f:
        return "".join(f.readlines()[-n:])

READY_MARKER = "Access the Web UI"   # offline.py logs this right before web.run_app() starts serving
TIMEOUT_S = 1800  # first run downloads the 7B model + STT model + compressor + embedding model; be generous
POLL_S = 5

start = time.time()
ready = False
while time.time() - start < TIMEOUT_S:
    if server_proc.poll() is not None:
        print(tail(LOG_PATH))
        raise RuntimeError(
            f"Server process exited early with return code {server_proc.returncode}. See log above."
        )
    if READY_MARKER in open(LOG_PATH).read():
        ready = True
        break
    time.sleep(POLL_S)

print(tail(LOG_PATH))

if not ready:
    raise TimeoutError(
        f"Server did not report readiness within {TIMEOUT_S}s. Check the log tail above -- "
        "the most common causes are a slow first-time model download or a CUDA/driver mismatch."
    )

print("\nServer is up and warmed up.")
print("Look for 'RAG index loaded: N chunks' and 'LoRA merged and unloaded' in the log tail above to confirm "
      "RAG and LoRA both initialized successfully -- RAG/LoRA/STT load failures are logged as errors but do "
      "NOT crash the server, it just falls back to the base conversational model, so don't assume they worked "
      "just because the server started.")


## 14. Expose the port & get the access URL

RunPod will not route external traffic to a port unless you explicitly expose it. In the pod's **Connect**
page, add an **HTTP Service** for `SERVER_PORT` (default `8998`) if you haven't already. RunPod then serves it
at `https://<POD_ID>-<PORT>.proxy.runpod.net`, with RunPod terminating TLS — which is exactly why
`USE_APP_TLS = False` is the right default here (see Section 12).


In [ ]:
pod_id = os.environ.get("RUNPOD_POD_ID")

if pod_id:
    public_url = f"https://{pod_id}-{SERVER_PORT}.proxy.runpod.net"
    print("RunPod public URL (requires SERVER_PORT to be exposed as an HTTP Service on the pod's Connect page):")
    print(" ", public_url)
else:
    print(
        "RUNPOD_POD_ID was not found in the environment. If you are on RunPod, expose "
        f"port {SERVER_PORT} as an HTTP Service from the pod's Connect page and use the proxy URL shown there."
    )

scheme = "https" if USE_APP_TLS else "http"
print(f"Local URL inside the pod: {scheme}://localhost:{SERVER_PORT}")


## 15. Verification — live pipeline smoke test (RAG + LoRA + tool calling)

`moshi/moshi/offline.py` is now a live WebSocket server (like the old `server.py`), not a batch WAV-in/WAV-out
tool — the old CLI flags (`--input-wav`, `--output-wav`, `--voice-prompt`) don't exist on it as top-level flags
anymore. Instead, this smoke test opens a real WebSocket connection to `/api/chat`, exactly like the web UI
does, and confirms the server gets all the way through session setup — model + Mimi + STT + LoRA + RAG index +
system prompt — and sends the handshake byte back. This exercises the full RAG/LoRA/tool-calling-aware startup
path without needing a browser or microphone.

The server only handles one session at a time (`ServerState.lock` in `offline.py`), so this test closes its
connection immediately after the handshake, freeing the server for the live web UI afterward.


In [ ]:
import asyncio

import aiohttp

async def _smoke_test():
    scheme = "wss" if USE_APP_TLS else "ws"
    connect_kwargs = {"ssl": False} if USE_APP_TLS else {}   # skip cert check for the app's self-signed cert
    url = f"{scheme}://localhost:{SERVER_PORT}/api/chat?voice_prompt=NATF2.pt"

    async with aiohttp.ClientSession() as session:
        async with session.ws_connect(url, timeout=30, **connect_kwargs) as ws:
            msg = await asyncio.wait_for(ws.receive(), timeout=30)
            assert msg.type == aiohttp.WSMsgType.BINARY, f"Unexpected message type: {msg.type}"
            assert msg.data == b"\x00", f"Expected handshake byte b'\\x00', got {msg.data!r}"
            print("Handshake byte received -- session setup (model + Mimi + STT + LoRA + RAG) succeeded.")

asyncio.run(_smoke_test())
print("\nLive pipeline smoke test succeeded.")


## 16. Verification — HTTP check on the live server

Confirms the running `moshi.offline` process answers on `/` (the web UI's `index.html`, served from `dist.tgz`).


In [ ]:
import ssl
import urllib.request

scheme = "https" if USE_APP_TLS else "http"
ctx = ssl._create_unverified_context() if USE_APP_TLS else None

with urllib.request.urlopen(f"{scheme}://localhost:{SERVER_PORT}/", context=ctx, timeout=15) as resp:
    print("HTTP status :", resp.status)
    print("Content-Type:", resp.headers.get("Content-Type"))
    assert resp.status == 200, f"Expected 200, got {resp.status}"

print("Web UI is being served correctly.")


## 17. Using the live web UI

Open the URL printed in Section 14 in a browser, allow microphone access when prompted (this is the one
browser permission step that can't be automated), and start talking.

### Try the RAG + tool-calling demo

Use the **Customer service example** role prompt below, then ask about something covered in `text.txt`
(e.g. *"How much is the compost bin add-on?"* or *"What happens if my pickup gets missed?"*) — watch the server
log (tail `LOG_PATH` from Section 13, e.g. re-run `print(tail(LOG_PATH))`) for `RAG CONTEXT WINDOW` entries
showing the retrieved passages. Ask something clearly outside `text.txt` (e.g. *"What's the weather in Paris
right now?"*) to see the web-search tool fire instead, if you enabled it in Section 11.

### Voices (from `README.md`)

```
Natural(female): NATF0, NATF1, NATF2, NATF3
Natural(male):   NATM0, NATM1, NATM2, NATM3
Variety(female): VARF0, VARF1, VARF2, VARF3, VARF4
Variety(male):   VARM0, VARM1, VARM2, VARM3, VARM4
```

### Example role prompts (from `README.md`)

- Assistant role: `You are a wise and friendly teacher. Answer questions or provide advice in a clear and
  engaging way.`
- Casual conversation: `You enjoy having a good conversation.`
- Customer service example (pairs with the `text.txt` RAG demo above): `You work for CitySan Services which is
  a waste management company and your name is Ayelen Lucero. Information: Verify customer name Omar Torres.
  Current schedule: every other week. Upcoming pickup: April 12th. Compost bin service available for
  $8/month add-on.`

See the repo's `README.md` "Prompting Guide" section for more examples and guidance.


## 18. GPU optimization notes for RTX 5090

- **bf16 by default**: `moshi/moshi/models/loaders.py:get_moshi_lm` loads the main LM in `torch.bfloat16`, and
  the STT model (`CheckpointInfo.get_moshi`) does the same. Blackwell has fast native bf16 Tensor Core
  throughput, so no dtype changes are needed.
- **VRAM budget on a 32GB RTX 5090** (everything shares the one GPU, since `--compressor-device` defaults to
  `cuda`): the 7B PersonaPlex LM in bf16 (~14GB) + two Mimi codec instances (small) + the ~1B STT model in bf16
  (~2GB) + its own Mimi + the 4-bit-quantized Qwen2.5-1.5B compressor (~1GB). The RAG embedding model
  (`sentence-transformers`) runs on **CPU** by design (see `ServerState.__init__` in `offline.py`), so it
  doesn't compete for VRAM. This comfortably fits with headroom for KV-cache and activations; `--cpu-offload`
  should not be needed.
- **`--cpu-offload` is unnecessary here**: it exists in `offline.py` for GPUs with insufficient VRAM (via
  `accelerate`'s `infer_auto_device_map`). Only enable `USE_CPU_OFFLOAD` if you're also running other large
  workloads on the same GPU.
- **One process = one GPU's worth of model**: `ServerState.__init__` loads the main LM, two Mimi instances, the
  STT model, and the compressor once per process and keeps them resident. Don't launch a second `moshi.offline`
  process against the same GPU unless you've confirmed there's VRAM headroom — check with `nvidia-smi`.
- **RAG and LoRA degrade gracefully, not loudly**: if the STT checkpoint, the LoRA adapter, or the compressor
  fail to load, `ServerState.__init__` logs a clear error and falls back to the base conversational model
  instead of crashing the server. After Section 13's poll cell, actually check the log tail for
  `RAG index loaded: N chunks` and `LoRA merged and unloaded` to confirm both came up — don't just assume it
  worked because the server started.
- **Turn-detection VAD is energy-based, not learned**: `offline.py` has no trained VAD head for the STT model,
  so turn-end and barge-in detection use simple audio RMS energy (`ServerState._pcm_silence_score`) against
  `--vad-energy-scale` (default `0.02`) instead. In a noisy room this can be less reliable than a learned VAD —
  raise `--vad-energy-scale` if background noise is triggering false turn-ends, or lower it if quiet speech
  isn't being detected as speech.
- **Warmup cost is already handled**: `state.warmup()` runs 4 dummy frames through the full encode → LM →
  decode path (main model and, if RAG loaded, the STT model) right after model load, ahead of any real
  connections — that's the per-process latency you saw the log wait for in Section 13, not something to
  optimize further.
- **CUDA build must match the GPU**: Blackwell (`sm_120`) needs the `cu130` PyTorch wheels installed in
  Section 5, and a recent enough `bitsandbytes` (>=0.45) for the compressor's 4-bit quantization to have
  Blackwell kernels. If you ever see `no kernel image is available for execution on the device`, re-run the
  relevant reinstall cell in Section 5 (something likely reinstalled an incompatible build afterward).


## 19. Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `401`/`403` downloading model assets | License not accepted, or `HF_TOKEN` doesn't belong to the account that accepted it | Re-check Section 7: accept the license at the model page with the **same** account whose token you pasted |
| `no kernel image is available for execution on the device` | Torch (or `bitsandbytes`) build doesn't have Blackwell (`sm_120`) kernels | Re-run the `cu130` reinstall cell and the `-U bitsandbytes` cell in Section 5, then re-run Section 6's smoke test |
| `ImportError` / build errors mentioning `opus` while installing `sphn` | `libopus-dev` missing | Re-run Section 3, then re-run Section 5 |
| Server process exits immediately, log shows a CUDA OOM | Not enough free VRAM (e.g. another process holds the GPU) | Check `nvidia-smi`; consider `USE_CPU_OFFLOAD = True` (Section 2) and re-run Sections 13-14 |
| Log shows `RAG disabled — STT model load failed` | The STT repo (`kyutai/stt-1b-en_fr-candle`) didn't download, or its checkpoint format doesn't match `CheckpointInfo`'s assumptions | Check the full error in the log; try a different PyTorch-format STT checkpoint repo via `STT_HF_REPO` in Section 8 if this one doesn't load. The server still runs as a plain conversational assistant either way |
| Log shows `RAG disabled — index load failed` or `0 chunks` | `text.txt` is empty, or the index build in Section 10 didn't actually run / failed | Re-run both cells in Section 10 and check the printed chunk count before starting the server |
| Log shows `LoRA load failed` or `No adapter_config.json found` | `moshi/lora` is missing `adapter_config.json`/`adapter_model.safetensors`, or `peft` isn't installed | Re-run Section 5's dependency cell; check Section 9's assertion passed |
| Web search never seems to fire | No key entered in Section 11, or the assistant's question already scores above `--web-search-trigger-below` locally | Re-run Section 11 with a real key, then re-run Sections 13-14; try a question clearly outside `text.txt` |
| Browser blocks microphone / `getUserMedia` fails | Page wasn't loaded over a secure context | Use the RunPod **proxy** URL from Section 14 (HTTPS at the edge), not a plain `http://<pod-ip>:8998` URL |
| Can't reach the URL from outside the pod at all | Port not exposed | In the RunPod console, add `SERVER_PORT` as an HTTP Service on the pod's Connect page |
| First launch seems to hang for several minutes | Normal — first run downloads multi-GB weights (main model, STT model, compressor) and extracts `voices.tgz`/`dist.tgz` | Watch the log tail printed by Section 13; increase `TIMEOUT_S` if your network is slow |
| `mkcert` warnings in the log | Only relevant when `USE_APP_TLS = True`; `moshi/moshi/utils/connection.py` falls back to plain HTTP automatically if `mkcert` can't be installed | Safe to ignore in default (proxy) mode |

### Recap: what genuinely cannot be automated by this notebook
1. Clicking "Agree and access repository" on the gated HF model page.
2. Issuing the HF access token for that account.
3. Getting a web-search API key, if you want tool calling enabled.
4. Exposing `SERVER_PORT` as an HTTP Service in the RunPod console.
5. The browser's microphone-permission prompt.


## 20. Stop the server (run only when you want to shut it down)

This is a management utility cell, **not** part of the linear startup flow — running it will terminate the
backend you just verified above. Run it deliberately when you're done with the session, not as part of a
top-to-bottom "Run All".


In [ ]:
# Intentionally NOT meant to run automatically as part of the startup sequence above.
try:
    server_proc.terminate()
    server_proc.wait(timeout=15)
    print(f"Server process {server_proc.pid} stopped.")
except NameError:
    print("No server_proc in scope -- nothing to stop.")
except subprocess.TimeoutExpired:
    server_proc.kill()
    print(f"Server process {server_proc.pid} killed after not stopping gracefully.")
